# Sarcasm Detection on SARC — Colab notebook

This notebook runs the project end-to-end on a free Colab GPU. It **imports the
code from `project/`** rather than duplicating it, so the logic stays in one
documented place (see the `docs/` folder).

**Before you start:** switch to a GPU runtime — *Runtime → Change runtime type → T4 GPU*.

## 1. Get the project code

Use **one** of the options in the next cell (clone a Git remote, or mount Google
Drive where you uploaded the repo), then set `PROJECT_DIR`.

In [ ]:
# Option A: clone from your Git remote (uncomment and fill in the URL):
# !git clone https://github.com/<you>/EmiliaProject.git /content/EmiliaProject

# Option B: mount Google Drive and point PROJECT_DIR at your uploaded copy:
# from google.colab import drive; drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/EmiliaProject'

PROJECT_DIR = "/content/EmiliaProject"

import os
assert os.path.isdir(PROJECT_DIR + "/project"), (
    f"Repo not found at {PROJECT_DIR}. Clone it or mount Drive first."
)
print("Found project at", PROJECT_DIR)

In [ ]:
# 2. Install dependencies (a couple of minutes the first time).
!pip install -q -r {PROJECT_DIR}/project/requirements.txt

## 3. Get the dataset

Upload your Kaggle API token (`kaggle.json`, from kaggle.com → Account → *Create
New API Token*) when prompted, and the SARC corpus is downloaded into `data/raw/`.

Alternatively, skip this cell and upload `train-balanced-sarcasm.csv` by hand into
`EmiliaProject/data/raw/`.

In [ ]:
import os
os.makedirs(f"{PROJECT_DIR}/data/raw", exist_ok=True)

from google.colab import files
print("Upload your kaggle.json ...")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d danofer/sarcasm -p {PROJECT_DIR}/data/raw --unzip
!ls -la {PROJECT_DIR}/data/raw

In [ ]:
# 4. Sanity-check the data pipeline (cleaning + splits + class balance).
!python {PROJECT_DIR}/project/run.py prepare

In [ ]:
# 5. Classical TF-IDF baseline, without and with thread context.
!python {PROJECT_DIR}/project/run.py baseline
!python {PROJECT_DIR}/project/run.py baseline --context

## 6. Fine-tune the transformers

Each run trains one (model, context) configuration. On a T4 with the default
150k-row subset, expect roughly tens of minutes per run. To smoke-test faster,
lower `DataConfig.subset_size` in `project/config.py`, or pass `--epochs 1`.

In [ ]:
# BERT — no context, then with context.
!python {PROJECT_DIR}/project/run.py train --model bert-base-uncased
!python {PROJECT_DIR}/project/run.py train --model bert-base-uncased --context

In [ ]:
# RoBERTa — no context, then with context.
!python {PROJECT_DIR}/project/run.py train --model roberta-base
!python {PROJECT_DIR}/project/run.py train --model roberta-base --context

In [ ]:
# 7. McNemar significance test: does context change BERT's predictions significantly?
!python {PROJECT_DIR}/project/run.py compare \
    --a {PROJECT_DIR}/results/bert-base-uncased_noctx_seed42_preds.npz \
    --b {PROJECT_DIR}/results/bert-base-uncased_ctx_seed42_preds.npz

In [ ]:
# 8. Aggregate every results/*_metrics.json into the comparison table.
!python {PROJECT_DIR}/project/run.py report

from IPython.display import Markdown, display
display(Markdown(open(f"{PROJECT_DIR}/results/summary.md").read()))